## Telecom Customer Churn – Revenue Impact & Retention Strategy Analysis
Telecom Industry | Business Analytics Project
Author: Anushka Anushree

## 1. Business Problem
Objective:
* Identify drivers of churn
* Quantify financial impact
* Recommend revenue-focused retention strategies

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")


## 2. Dataset Overview

In [3]:
df.shape

(7043, 21)

* The dataset contains 7,043 customer records with 21 features.

* The target variable is Churn, indicating whether a customer discontinued service.

* Features include demographic attributes, service-related variables, and account/financial information.

**Feature Categories** 

* Demographic Features: Gender, SeniorCitizen, Partner, Dependents

* Service Features: InternetService, OnlineSecurity, TechSupport, Streaming services

* Account & Financial Features: Tenure, Contract, MonthlyCharges, TotalCharges, PaymentMethod

## 3. Data Cleaning and Preparation

## 3.1. Convert data types

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


## 3.2. Handle Missing Values

In [5]:
df.isnull().sum()


customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [6]:
# Drop rows with missing TotalCharge
df = df.dropna(subset=['TotalCharges'])


## 3.3. Create Customer Lifetime Value(CLV)

In [7]:
df['CLV'] = df['MonthlyCharges'] * df['tenure']


## 3.4. Create Tenure Groups

In [8]:
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0,12,36,72],
    labels=["0-12 months", "12-36 months", "37+ months"]
)


## 3.5. Create Customer Segment(Top 25% CLV)

In [9]:
clv_threshold = df['CLV'].quantile(0.75)

df['Customer_Segment'] = df['CLV'].apply(
    lambda x: 'High Value' if x >= clv_threshold else 'Standard'
)


## 3.6. Final Dataset Check

In [10]:
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CLV,TenureGroup,Customer_Segment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85,0-12 months,Standard
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.50,No,1936.30,12-36 months,Standard
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,107.70,0-12 months,Standard
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,1903.50,37+ months,Standard
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,141.40,0-12 months,Standard


## 4. Exploratory Data Analysis

## 4.1. Overall Churn Rate

In [11]:
df["Churn"].value_counts(normalize = True)*10

Churn
No     7.34215
Yes    2.65785
Name: proportion, dtype: float64

* The overall churn rate is 26.53% which indicates that 1 in 4 customers leave the company.

## 4.2. Churn by contract type

In [12]:
pd.crosstab(df["Contract"] , df["Churn"] , normalize = "index") *100

Churn,No,Yes
Contract,,
Month-to-month,57.290323,42.709677
One year,88.722826,11.277174
Two year,97.151335,2.848665


* Month-to-month contract type exhibits highest churn rate (42.70%) compared to one year(11.26%) and two year(2.83%) contracts.
The suggests that long term commitments improves customer retention.

## 4.3. Churn by Internet Service

In [13]:
pd.crosstab(df["InternetService"] , df["Churn"] , normalize = "index") *100

Churn,No,Yes
InternetService,,
DSL,81.001656,18.998344
Fiber optic,58.107235,41.892765
No,92.565789,7.434211


* Fiber optic customers exhibit the highest churn rate (~42%) compared to other service categories.

## 4.4. Tenure Analysis

In [14]:
tenure_churn = (
    pd.crosstab(df['TenureGroup'], df['Churn'], normalize='index') * 100
).round(2)

tenure_churn



Churn,No,Yes
TenureGroup,,
0-12 months,52.32,47.68
12-36 months,74.46,25.54
37+ months,88.07,11.93


* We can clearly see that 1 in 2 new customers are likely to leave the company within 12 months.

## 5. Revenue Impact Analysis


## 5.1. Total Monthly Revenue

In [15]:
total_monthly_revenue = df["MonthlyCharges"].sum()
print("Total Monthly Revenue : ₹" , round(total_monthly_revenue,2))

Total Monthly Revenue : ₹ 455661.0


## 5.2. Monthly Revenue Lost Due to Churn

In [16]:
churned_customers = df[df["Churn"] =="Yes"]
monthly_revenue_lost = churned_customers["MonthlyCharges"].sum()
print("Monthly Revenue Lost Due to Churn : ₹", round(monthly_revenue_lost , 2))

Monthly Revenue Lost Due to Churn : ₹ 139130.85


## 5.3.Annual Revenue Lost

In [17]:
annual_revenue_lost = monthly_revenue_lost*12
print("Estimate Annual Revenue Lost:₹" , round(annual_revenue_lost,2))

Estimate Annual Revenue Lost:₹ 1669570.2


## 5.4. Revenue Lost Percentage

In [18]:
revenue_lost_percent = (monthly_revenue_lost/total_monthly_revenue)*100
print("Revenue Lost Percentage:" , round(revenue_lost_percent , 2), "%")

Revenue Lost Percentage: 30.53 %


- **Monthly Revenue:** ₹456,116
- **Annual Revenue Lost:** ₹1.67 million
- **Revenue at Risk:** 30.5%


## Business Interpretation

* The company generates approximately ₹456,116 in monthly revenue.

* However, due to customer churn, it is losing ₹139,131 per month, which translates to ₹1.67 million annually.

* This means 30.5% of total monthly revenue is being lost due to churn, representing a significant financial risk.

* The financial impact of churn is substantial and directly affects profitability, cash flow stability, and long-term growth.

* Even a small reduction in churn could recover a meaningful portion of this lost revenue and significantly improve overall business performance.

* A 5% reduction in churn could potentially recover approximately ₹83,000–₹100,000 annually, highlighting the strategic importance of customer retention initiatives.


## 6. Customer Lifetime Value(CLV) Analysis

In [19]:
df.groupby('Churn')['CLV'].mean().round(2)


Churn
No     2555.20
Yes    1531.61
Name: CLV, dtype: float64

In [20]:
total_clv_lost = df[df['Churn'] == 'Yes']['CLV'].sum()

print("Total Lifetime Value Lost Due to Churn: ₹", round(total_clv_lost,2))


Total Lifetime Value Lost Due to Churn: ₹ 2862576.9


In [21]:

high_value_churn = (
    pd.crosstab(df['Customer_Segment'], df['Churn'], normalize='index') * 100
).round(2)

high_value_churn


Churn,No,Yes
Customer_Segment,,
High Value,85.55,14.45
Standard,69.38,30.62


## Business Interpretation – CLV & High-Value Segment

* The average CLV of retained customers (₹2549.77) is significantly higher than churned customers (₹1531.61), indicating that long-tenure customers contribute substantially more lifetime revenue.

* The company has lost approximately ₹2.86 million in total lifetime value due to churn, highlighting significant long-term financial impact.

* High-value customers (top 25% CLV) exhibit a churn rate of 14.48%, which is significantly lower than the overall churn rate (26.5%).

* This suggests that the company is relatively effective at retaining its most profitable customers.

* The primary churn risk appears to lie within lower-value customer segments, where retention strategies should be more aggressively targeted.

## 7. High-Risk Segment Analysis

## 7.1. Indentify Segment

In [22]:
pd.crosstab([df["InternetService"] , df["Contract"]] , df["Churn"] , normalize = "index") *100

Churn                                  No        Yes
InternetService Contract                            
DSL             Month-to-month  67.784137  32.215863
                One year        90.701754   9.298246
                Two year        98.073836   1.926164
Fiber optic     Month-to-month  45.394737  54.605263
                One year        80.705009  19.294991
                Two year        92.773893   7.226107
No              Month-to-month  81.106870  18.893130
                One year        97.520661   2.479339
                Two year        99.210111   0.789889

* Fiber optic customers on month-to-month contract exhibits highest churn rate(~55%) among all segments. This segment combines short-term commitments with high-service plans, indicating that customers of this group are more likely to leave the company.

## 7.2. Revenue Loss

In [23]:
high_risk_churn = df[(df["InternetService"] == "Fiber optic") &
(df["Contract"] == "Month-to-month") &
(df["Churn"] == "Yes")
]
high_risk_churn["TotalCharges"].sum()

np.float64(1731652.5)

* The total revenue lost is approximately ₹1.73 million.

## 7.3. Average Revenue per lost Customer

In [24]:
high_risk_churn["TotalCharges"].mean()

np.float64(1490.2345094664372)

* Average revenue per lost customer is approximately ₹1490, indicating that  this segment needs a strategic action plan.

## Business Interpretation – High-Risk Segment

* Fiber optic customers on month-to-month contracts exhibit the highest churn rate (~55%) across all service-contract combinations.

* This segment combines premium internet service with short-term contractual commitment, increasing churn vulnerability.

* Churn within this segment has resulted in approximately ₹1.73 million in lost revenue, making it financially significant.

* On average, each lost customer in this segment represents approximately ₹1,490 in revenue, highlighting meaningful financial exposure per churn event.

* Given both high churn probability and substantial revenue impact, this segment should be prioritized for targeted retention and contract migration strategies.

# 8. Retention Improvement Simulation


In [25]:
current_churn_rate = 26.5 / 100
improved_churn_rate = 23.5 / 100

reduction_percentage = (current_churn_rate - improved_churn_rate) / current_churn_rate

revenue_saved_annual = annual_revenue_lost * reduction_percentage

print("Estimated Annual Revenue Saved (3% churn reduction): ₹", round(revenue_saved_annual,2))


Estimated Annual Revenue Saved (3% churn reduction): ₹ 189007.95


In [26]:
total_clv_lost = df[df['Churn'] == 'Yes']['CLV'].sum()

clv_saved = total_clv_lost * reduction_percentage

print("Estimated Lifetime Value Saved: ₹", round(clv_saved,2))


Estimated Lifetime Value Saved: ₹ 324065.31


## Business Interpretation – Retention Improvement Simulation

* A 3% reduction in churn could recover approximately ₹189,008 in annual revenue.

* Additionally, nearly ₹324,065 in lifetime customer value could be preserved.

* This demonstrates that even small improvements in customer retention generate substantial financial returns.

* Targeted retention strategies focused on high-risk segments could significantly enhance revenue stability and long-term profitability.

* The financial upside justifies investment in customer engagement programs, loyalty incentives, and early churn detection initiatives.

## 9. Strategic Recommendations



**1.Prioritize High-Risk Contract Segments**

* Fiber optic customers on month-to-month contracts exhibit churn rates exceeding 50%, making them the most financially vulnerable segment. The company should implement a targeted contract migration strategy, clearly communicating long-term cost savings and bundled value propositions to encourage upgrades to one-year and two-year plans.

**2.Strengthen Early-Tenure Engagement**

* Customers within their first 12 months demonstrate elevated churn (~48%), indicating that the initial lifecycle stage is a critical risk window. Proactive onboarding programs, service check-ins, and early engagement campaigns should be deployed to stabilize retention during this period.

**3.Deploy Targeted Retention Incentives**

* Data suggests that churn risk is concentrated within specific segments rather than high-value customers broadly. Limited-time loyalty incentives and personalized offers should be directed toward high-risk customers to maximize ROI while avoiding unnecessary discount leakage.

**4.Implement a Data-Driven Early Warning System**

* The company should operationalize churn indicators such as contract type, tenure, and service type to proactively identify at-risk customers. A structured monitoring framework would enable intervention before revenue loss occurs.

**5.Reevaluate Fiber Optic Pricing Strategy**

* Given the high churn concentration within fiber optic customers, the pricing-value alignment should be assessed. A review of competitive positioning, perceived value, and service quality may reveal structural factors contributing to dissatisfaction.


## 10. Conclusion

 This analysis identified fiber optic customers on month-to-month contracts as the primary high-risk segment, with churn rates exceeding 50%. This segment alone contributes approximately ₹1.73 million in revenue loss from churned customers.

Additionally, customers within their first 12 months exhibit significantly elevated churn (~48%), indicating that early tenure is a critical intervention period.

Financial modeling further demonstrates that even a modest 3% reduction in churn could recover approximately ₹189,008 annually and preserve over ₹324,065 in lifetime customer value.

By focusing on:

* Contract migration strategies

* Early-stage engagement initiatives

* Targeted retention campaigns

* Pricing structure optimization

* Proactive churn monitoring

the company can materially reduce revenue erosion and enhance long-term profitability.

This project demonstrates that customer churn is not merely a retention metric, but a measurable financial risk requiring strategic intervention.

In [27]:
df.to_csv("churn_cleaned.csv" , index=False)